# Cleaning the dataset


### Variable overview
* Raw_df: the unchanged dataframe
* df: the dataframe with irrelevant columns dropped

### Dataset cleaning pipeline overview

Input: `stylecom_raw.csv`
Pipeline runs in 5 stages:

- **1. Setup**: import dependencies and load the raw CSV with `encoding="latin-1"`
- **2. Initial inspection**: check shape, dtypes, nulls, and sample reviews to understand the data
- **3. Clean text columns**: the bulk of the work:
  - detect suspicious tokens (broken accents, replacement characters, split contractions)
  - apply `ftfy.fix_text` as a first pass, then layer four regex maps (control chars, names/brands, contractions, French fashion vocabulary) and two literal maps (direct replacements, word-bridge fragments)
  - write cleaned versions as `designer_clean` and `review_clean`
- **4. Export**: write `data/stylecom_cleaned.csv` (UTF-8)
- **5. Quality checks**: check the remaining suspicious tokens

# 1. Setup

In [1]:
# Cell 1: imports
import pandas as pd
import textwrap
from ftfy import fix_text
import re
from collections import Counter
from pathlib import Path

In [2]:
# Cell 2: file paths
PATH = "data/stylecom_raw.csv"
CLEANING_PATH = Path("cleaning_outputs")
CLEANING_PATH.mkdir(exist_ok=True)

# 2. Initial inspection

In [3]:
# Cell 3: load raw CSV and inspect
raw_df = pd.read_csv(PATH, encoding="latin-1")

raw_df.shape
raw_df.info()
raw_df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 6629 entries, 0 to 6628
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   year        6629 non-null   int64
 1   season      6629 non-null   str  
 2   designer    6629 non-null   str  
 3   author      6492 non-null   str  
 4   city        6629 non-null   str  
 5   date        6629 non-null   str  
 6   image       6629 non-null   str  
 7   review      6629 non-null   str  
 8   Unnamed: 8  5 non-null      str  
dtypes: int64(1), str(8)
memory usage: 466.2 KB


year             0
season           0
designer         0
author         137
city             0
date             0
image            0
review           0
Unnamed: 8    6624
dtype: int64

In [4]:
# Cell 4: preview raw data
raw_df.head()

,year,season,designer,author,city,date,image,review,Unnamed: 8
0,2000,Spring,Matt Nye,Armand Limnander,NEW YORK,17-Sep-99,"<div class=""image-container""><a class=""slidesh...",Designer Matt Nye's sophomore show featured a ...,NaN
1,2000,Spring,Giorgio Armani,Armand Limnander,MILAN,29-Sep-99,"<div class=""image-container""><a class=""slidesh...","Armani proposed a light, feminine silhouette f...",NaN
2,2000,Spring,Eric Bergre,Armand Limnander,PARIS,4-Oct-99,"<div class=""image-container""><a class=""slidesh...",Broadway Garnier was the theme for Eric Berg?r...,NaN
3,2000,Spring,C_line,Armand Limnander,PARIS,7-Oct-99,"<div class=""image-container""><a class=""slidesh...",Getaway glamour was the theme for Celine's str...,NaN
4,2000,Spring,Byblos,Armand Limnander,MILAN,27-Sep-99,"<div class=""image-container""><a class=""slidesh...",Judo Jetson blends my favorite cartoon charact...,NaN


# 3. Cleaning

## Dropping uninformative columns

In [5]:
# Cell 5: drop uninformative columns
df = raw_df.drop(columns=["Unnamed: 8", "image"])
df.head()

,year,season,designer,author,city,date,review
0,2000,Spring,Matt Nye,Armand Limnander,NEW YORK,17-Sep-99,Designer Matt Nye's sophomore show featured a ...
1,2000,Spring,Giorgio Armani,Armand Limnander,MILAN,29-Sep-99,"Armani proposed a light, feminine silhouette f..."
2,2000,Spring,Eric Bergre,Armand Limnander,PARIS,4-Oct-99,Broadway Garnier was the theme for Eric Berg?r...
3,2000,Spring,C_line,Armand Limnander,PARIS,7-Oct-99,Getaway glamour was the theme for Celine's str...
4,2000,Spring,Byblos,Armand Limnander,MILAN,27-Sep-99,Judo Jetson blends my favorite cartoon charact...


### Helper function

In [6]:
# Cell 6: view_reviews helper and sample
def view_reviews(df, column):
    for text in df[column].sample(5):
        print(textwrap.fill(text, width=80))
        print("-" * 80)

view_reviews(df, "review")

With a palette centered mainly around black and white, Oscar de la Renta offered
a more somber take than usual on his frothy classics, but still provided a wide
range of stylistic options for his faithful clientele.  For lunch at Swifty's,
there were salt-and-pepper tweed suits, gently flared skirts to the knee, and
wool coats with fur collars and prim leather bow belts. De la Renta's vision of
casual luxury also included cashmere and boiled wool cardigan jackets, ruffled
silk charmeuse blouses and even a sexy, youthful fuchsia shearling mini. Evening
took a turn for the dramatic with beaded and feathered slipdresses, tapestry-
print velvet trousers and gowns, and a couple of decidedly gothic black opera
coats. Some of these looks will prove overwhelming for many, but steadfast Oscar
fans will still be able to find plenty of their favorite ruffled skirts and sexy
halter tops.  Shoe and bag fetishists will be delighted to know that de la Renta
launched his signature line of accessories 

## Checking broken character encoding

In [7]:
# Cell 7: suspicious token detection
TEXT_COLS = ["designer", "review"]


def suspicious_tokens(text):
    """
    Find all tokens containing ?, _, or C1/control-byte artifacts from latin1 read.
    """
    if not isinstance(text, str):
        return []
    # catches tokens containing ?, _, or C1/control-byte artifacts from latin1 read
    return re.findall(r"\b\S*[?_-]\S*\b", text)

def count_suspicious_tokens(df, text_cols):
    """
    Count suspicious tokens across specified text columns in the DataFrame.
    """
    token_counts = Counter()
    for col in text_cols:
        for text in df[col].dropna():
            token_counts.update(suspicious_tokens(text))

    suspect_df = (
        pd.DataFrame(token_counts.items(), columns=["token", "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    return suspect_df

suspect_df = count_suspicious_tokens(df, TEXT_COLS)
suspect_df.head(200)

suspect_df.to_csv("cleaning_outputs/suspect_tokens.csv", index=False)
print(len(suspect_df))

10347


## Fixing broken character encoding

In [8]:
# Cell 8: ftfy encoding fix
df["review"] = df["review"].apply(fix_text)

### Mappings

In [9]:
# Cell 9: replacement maps
CONTROL_CHAR_MAP = {
    "\x8f": "?",
    "\x8d": "?",
    "\x90": "?",
    "\x9d": "?",
    "\u00e3": " ",
    "\u00e9": "é",
    "\u00e9": "'",
    "\u00ef": "ï"
}

NAME_MAP = {
    r"(?<!\w)Berg\?re(?!\w)": "Bergère",
    r"(?<!\w)Bergre(?!\w)": "Bergère",
    r"(?<!\w)Valer\?e(?!\w)": "Valérie",
    r"(?<!\w)Val\?rie(?!\w)": "Valérie",

    r"(?<!\w)Chlo\?(?!\w)": "Chloé",
    r"(?<!\w)Chlo_(?!\w)": "Chloé",
    r"(?<!\w)Chlo\?'s(?!\w)": "Chloé's",

    r"(?<!\w)C[\?_]line(?!\w)": "Céline",
    r"(?<!\w)C\?line(?!\w)": "Céline",
    r"(?<!\w)C_line(?!\w)": "Céline",

    r"(?<!\w)Ferr\?(?!\w)": "Ferré",
    r"(?<!\w)Ferr_(?!\w)": "Ferré",
    r"(?<!\w)Ferr\?'s(?!\w)": "Ferré's",
    r"(?<!\w)Ferr_'s(?!\w)": "Ferré's",

    r"(?<!\w)Abaet\?(?!\w)": "Abaeté",
    r"(?<!\w)Abaet_(?!\w)": "Abaeté",

    r"(?<!\w)R\?yes(?!\w)": "Reyes",

    r"(?<!\w)Azrou\?l(?!\w)": "Azrouël",
    r"(?<!\w)Azrou\?l's(?!\w)": "Azrouël's",

    r"(?<!\w)K\?hne(?!\w)": "Kühne",
    r"(?<!\w)K\?hne's(?!\w)": "Kühne's",

    r"(?<!\w)St\?rk(?!\w)": "Stärk",
    r"(?<!\w)St\?rk's(?!\w)": "Stärk's",

    r"(?<!\w)Comme des Gar\?ons(?!\w)": "Comme des Garçons",
    r"(?<!\w)Comme des Garons(?!\w)": "Comme des Garçons",

    r"(?<!\w)Herm\?s(?!\w)": "Hermès",
    r"(?<!\w)Herms(?!\w)": "Hermès",
    r"(?<!\w)L'Or\?al(?!\w)": "L'Oréal",

    r"(?<!\w)Herv\? L\?ger(?!\w)": "Hervé Léger",
    r"(?<!\w)Herv_ L_ger(?!\w)": "Hervé Léger",
    r"(?<!\w)Herv_ L_ger by Max Azria(?!\w)": "Hervé Léger by Max Azria",
    r"(?<!\w)L\?ger(?!\w)": "Léger",
    r"(?<!\w)L_ger(?!\w)": "Léger",

    r"(?<!\w)V\?ronique(?!\w)": "Véronique",
    r"(?<!\w)V_ronique(?!\w)": "Véronique",
    r"(?<!\w)V\?ronique Leroy(?!\w)": "Véronique Leroy",

    r"(?<!\w)D\?tacher(?!\w)": "Détacher",
    r"(?<!\w)D\?tacher's(?!\w)": "Détacher's",
    r"(?<!\w)A D_tacher(?!\w)": "A Détacher",

    r"(?<!\w)Rhi\?(?!\w)": "Rhié",
    r"(?<!\w)Rhi_(?!\w)": "Rhié",

    r"(?<!\w)Mus\?e(?!\w)": "Musée",
    r"(?<!\w)Mus_e(?!\w)": "Musée",

    r"(?<!\w)Beyonc\?(?!\w)": "Beyoncé",

    r"(?<!\w)Ghesqui\?re(?!\w)": "Ghesquière",
    r"(?<!\w)Ghesquire(?!\w)": "Ghesquière",
    r"(?<!\w)Ghesqui\?re's(?!\w)": "Ghesquière's",
    r"(?<!\w)Ghesquire's(?!\w)": "Ghesquière's",

    r"(?<!\w)Gar\?ons(?!\w)": "Garçons",
    r"(?<!\w)Garons(?!\w)": "Garçons",
    r"(?<!\w)gar\?on(?!\w)": "garçon",

    r"(?<!\w)Ala\?a(?!\w)": "Alaïa",
    r"(?<!\w)Ala\?a-esque(?!\w)": "Alaïa-esque",
    r"(?<!\w)Ala\?a-inspired(?!\w)": "Alaïa-inspired",

    r"(?<!\w)Th\?allet(?!\w)": "Théallet",
    r"(?<!\w)Th\?allet's(?!\w)": "Théallet's",

    r"(?<!\w)Louren\?o(?!\w)": "Lourenço",
    r"(?<!\w)Loureno(?!\w)": "Lourenço",
    r"(?<!\w)Louren\?o's(?!\w)": "Lourenço's",
    r"(?<!\w)Loureno's(?!\w)": "Lourenço's",

    r"(?<!\w)Gr\?s(?!\w)": "Grès",
    r"(?<!\w)Courr\?ges(?!\w)": "Courrèges",
    r"(?<!\w)Fran\?oise(?!\w)": "Françoise",
    r"(?<!\w)Fran\?ois(?!\w)": "François",
    r"(?<!\w)Herv\?(?!\w)": "Hervé",
    r"(?<!\w)S\?bastien(?!\w)": "Sébastien",
    r"(?<!\w)Ren\?e(?!\w)": "Renée",
    r"(?<!\w)C\?dric(?!\w)": "Cédric",
    r"(?<!\w)C_dric(?!\w)": "Cédric",
    r"(?<!\w)Bj\?rk(?!\w)": "Björk",
    r"(?<!\w)Joaqu\?n(?!\w)": "Joaquín",
    r"(?<!\w)Ram\?n(?!\w)": "Ramón",
    r"(?<!\w)F\?lix(?!\w)": "Félix",
    r"(?<!\w)Mar\?a(?!\w)": "María",
    r"(?<!\w)Ana\?s(?!\w)": "Anaïs",

    r"(?<!\w)Lyc\?e(?!\w)": "Lycée",
    r"(?<!\w)Op\?ra(?!\w)": "Opéra",
    r"(?<!\w)Citro\?n(?!\w)": "Citroën",
    r"(?<!\w)Almod\?var(?!\w)": "Almodóvar",
    r"(?<!\w)Ladur\?e(?!\w)": "Ladurée",
    r"(?<!\w)O\?Connor(?!\w)": "O'Connor",
    r"(?<!\w)Hy\?res(?!\w)": "Hyères",
    r"(?<!\w)Elys\?e(?!\w)": "Elysée",

    r"(?<!\w)H\?tel(?!\w)": "Hôtel",
    r"(?<!\w)h\?tel(?!\w)": "hôtel",

    r"(?<!\w)R\?camier(?!\w)": "Récamier",
    r"(?<!\w)S\?o(?!\w)": "São",
    r"(?<!\w)B\?nstr\?m(?!\w)": "Bönström",
    r"(?<!\w)B_nstr_m(?!\w)": "Bönström",
    r"(?<!\w)Fr_d_ric(?!\w)": "Frédéric",

    r"(?<!\w)Yigal AzrouФl(?!\w)": "Yigal Azrouël",
    r"(?<!\w)Kai KЩhne(?!\w)": "Kai Kühne",
    r"(?<!\w)St_rk(?!\w)": "Stärk",
    r"(?<!\w)Comme des GarЌons(?!\w)": "Comme des Garçons",
    r"(?<!\w)HermЏs(?!\w)": "Hermès",

    r"(?<!\w)Werkst\?tte(?!\w)": "Werkstätte",
    r"(?<!\w)B\?ndchen(?!\w)": "Bändchen",
    r"(?<!\w)D_coratifs(?!\w)": "Décoratifs",
    "Kitsun_": "Kitsuné",
}

CONTRACTION_MAP = {
    r"(?<=\w)n\?t(?!\w)": "n't",
    r"(?<=\w)\?re(?!\w)": "'re",
    r"(?<=\w)\?ve(?!\w)": "'ve",
    r"(?<=\w)\?ll(?!\w)": "'ll",
    r"(?<=\w)\?d(?!\w)": "'d",
    r"(?<=\w)\?m(?!\w)": "'m",
    r"(?<=\w)\?s(?!\w)": "'s",
}

DOMAIN_LANGUAGE_MAP = {
    r"(?<!\w)neglig\?e(?!\w)": "négligée",

    r"(?<!\w)d\?collet\?(?!\w)": "décolleté",
    r"(?<!\w)d\?collet\?s(?!\w)": "décolletés",
    r"(?<!\w)d\?collet\?e(?!\w)": "décolletée",
    r"(?<!\w)d\?colletage(?!\w)": "décolletage",

    r"(?<!\w)lingerie(?!\w)": "lingerie",

    r"(?<!\w)boucl\?(?!\w)": "bouclé",
    r"(?<!\w)boucl_(?!\w)": "bouclé",
    r"(?<!\w)boucl\?s(?!\w)": "bouclés",

    r"(?<!\w)piqu\?(?!\w)": "piqué",
    r"(?<!\w)piqu_(?!\w)": "piqué",

    r"(?<!\w)lam\?(?!\w)": "lamé",
    r"(?<!\w)lam_(?!\w)": "lamé",
    r"(?<!\w)lam\?s(?!\w)": "lamés",
    r"(?<!\w)Lam\?s(?!\w)": "Lamés",

    r"(?<!\w)cr\?pe(?!\w)": "crêpe",
    r"(?<!\w)velour(?!\w)": "velour",
    r"(?<!\w)velours(?!\w)": "velours",

    r"(?<!\w)appliqu\?(?!\w)": "appliqué",
    r"(?<!\w)appliqu\?s(?!\w)": "appliqués",
    r"(?<!\w)appliqu_d(?!\w)": "appliquéd",
    r"(?<!\w)appliqu\?d(?!\w)": "appliquéd",
    r"(?<!\w)appliqu_s(?!\w)": "appliqués",
    r"(?<!\w)appliqu_(?!\w)": "appliqué",
    r"(?<!\w)appliqu_ing(?!\w)": "appliquéing",

    r"(?<!\w)pliss\?(?!\w)": "plissé",
    r"(?<!\w)pliss\?e(?!\w)": "plissée",
    r"(?<!\w)pliss\?s(?!\w)": "plissés",
    r"(?<!\w)pliss_(?!\w)": "plissé",

    r"(?<!\w)matelass\?(?!\w)": "matelassé",

    r"(?<!\w)prot\?g\?(?!\w)": "protégé",
    r"(?<!\w)prot\?g\?e(?!\w)": "protégée",

    r"(?<!\w)cach\?(?!\w)": "caché",
    r"(?<!\w)cach\?e(?!\w)": "cachée",

    r"(?<!\w)haute coutur\?(?!\w)": "haute couture",
    r"(?<!\w)pr\?t-\?-porter(?!\w)": "prêt-à-porter",

    r"(?<!\w)touch\?(?!\w)": "touché",
    r"(?<!\w)entr\?e(?!\w)": "entrée",

    r"(?<!\w)appliqu[?éèêë]e(?!\w)": "appliquée",
    r"(?<!\w)appliqu[?éèêë]es?(?!\w)": "appliquées",
    r"(?<!\w)d[?éèêë]collet[?éèêë](?!\w)": "décolleté",
    r"(?<!\w)d[?éèêë]collet[?éèêë]s(?!\w)": "décolletés",

    r"(?<!\w)soign_(?!\w)": "soigné",
    r"(?<!\w)ombr_(?!\w)": "ombré",
    r"(?<!\w)ombr\?-dyed(?!\w)": "ombré-dyed",
    r"(?<!\w)ombr\?d(?!\w)": "ombréd",

    r"(?<!\w)d\?j(?!\w)": "déjà",
    r"(?<!\w)tr\?s(?!\w)": "très",
    r"(?<!\w)trs(?!\w)": "très",
    r"(?<!\w)apr\?s-ski(?!\w)": "après-ski",
    r"(?<!\w)m\?lange(?!\w)": "mélange",
    r"(?<!\w)m_lange(?!\w)": "mélange",
    r"(?<!\w)m\?langes(?!\w)": "mélanges",
    r"(?<!\w)mise-en-sc\?ne(?!\w)": "mise-en-scène",

    r"(?<!\w)na\?ve(?!\w)": "naïve",
    r"(?<!\w)na\?f(?!\w)": "naïf",
    r"(?<!\w)na\?vet(?!\w)": "naïveté",
    r"(?<!\w)na«vet_(?!\w)": "naïveté",

    r"(?<!\w)m\?tier(?!\w)": "métier",
    r"(?<!\w)ma\?tresse(?!\w)": "maîtresse",
    r"(?<!\w)si\?cle(?!\w)": "siècle",
    r"(?<!\w)minaudi\?re(?!\w)": "minaudière",
    r"(?<!\w)minaudi\?res(?!\w)": "minaudières",
    r"(?<!\w)f\?te(?!\w)": "fête",
    r"(?<!\w)marini\?re(?!\w)": "marinière",
    r"(?<!\w)cr\?me(?!\w)": "crème",
    r"(?<!\w)gr\?ce(?!\w)": "grâce",
    r"(?<!\w)pi\?ce(?!\w)": "pièce",
    r"(?<!\w)r\?sistance(?!\w)": "résistance",
    r"(?<!\w)r\?sum(?!\w)": "résumé",
    r"(?<!\w)caf\?s(?!\w)": "cafés",
    r"(?<!\w)soir\?e(?!\w)": "soirée",
    r"(?<!\w)fin-de-si\?cle(?!\w)": "fin-de-siècle",
    r"(?<!\w)Proven\?al(?!\w)": "Provençal",
    r"(?<!\w)outr_(?!\w)": "outré",
    r"(?<!\w)risqu_(?!\w)": "risqué",
    r"(?<!\w)dor\?e(?!\w)": "dorée",
    r"(?<!\w)pr\?cis(?!\w)": "précis",
    r"(?<!\w)ing\?nue(?!\w)": "ingénue",

    r"(?<!\w)d'\?tre(?!\w)": "d'être",
    r"(?<!\w)d'tre(?!\w)": "d'être",

    r"(?<!\w)d\?vor(?!\w)": "dévoré",
    r"(?<!\w)d_vor_(?!\w)": "dévoré",
    r"(?<!\w)devor_(?!\w)": "dévoré",
    r"(?<!\w)devor_s(?!\w)": "dévorés",

    r"(?<!\w)d\?grad(?!\w)": "dégradé",
    r"(?<!\w)d_grad_(?!\w)": "dégradé",

    r"(?<!\w)clich\?d(?!\w)": "clichéd",
    r"(?<!\w)clich_(?!\w)": "cliché",
    r"(?<!\w)clich\?s(?!\w)": "clichés",

    r"(?<!\w)gar\?on(?!\w)": "garçon",
    r"(?<!\w)piqu\?(?!\w)": "piqué",
    r"(?<!\w)Mus\?e(?!\w)": "Musée",
    r"(?<!\w)Mus_e(?!\w)": "Musée",
    r"(?<!\w)H\?tel(?!\w)": "Hôtel",
    r"(?<!\w)h\?tel(?!\w)": "hôtel",
}

DIRECT_MAP = {
    # names / brands
    "Ghesquière": "Ghesquière",
    "Ghesqui?re": "Ghesquière",
    "Ghesqui?re's": "Ghesquière's",
    "Ghesquire": "Ghesquière",
    "Ghesquire's": "Ghesquière's",

    "Herm?s": "Hermès",
    "Herms": "Hermès",

    "Gar?ons": "Garçons",
    "Garons": "Garçons",
    "gar?on": "garçon",

    "Azrou?l": "Azrouël",
    "Azrou?l's": "Azrouël's",

    "Ala?a": "Alaïa",
    "Ala?a-esque": "Alaïa-esque",
    "Ala?a-inspired": "Alaïa-inspired",

    "L?ger": "Léger",
    "L_ger": "Léger",

    "Th?allet": "Théallet",
    "Th?allet's": "Théallet's",

    "Louren?o": "Lourenço",
    "Louren?o's": "Lourenço's",
    "Loureno": "Lourenço",
    "Loureno's": "Lourenço's",

    "Gr?s": "Grès",
    "Val_rie": "Valérie",
    "C?line": "Céline",
    "C_line": "Céline",
    "St?rk": "Stärk",
    "St?rk's": "Stärk's",
    "K?hne": "Kühne",
    "K?hne's": "Kühne's",
    "Courr?ges": "Courrèges",
    "Val?rie": "Valérie",
    "Fran?oise": "Françoise",
    "Fran?ois": "François",
    "D?tacher": "Détacher",
    "D?tacher's": "Détacher's",
    "V?ronique": "Véronique",
    "V_ronique": "Véronique",
    "Chlo?": "Chloé",
    "Chlo_": "Chloé",
    "Chlo?'s": "Chloé's",
    "Herv?": "Hervé",
    "Herv_": "Hervé",
    "S?bastien": "Sébastien",
    "Ren?e": "Renée",
    "C?dric": "Cédric",
    "C_dric": "Cédric",
    "Bj?rk": "Björk",
    "Joaqu?n": "Joaquín",
    "Ram?n": "Ramón",
    "F?lix": "Félix",
    "Mar?a": "María",
    "Lyc?e": "Lycée",
    "Op?ra": "Opéra",
    "Citro?n": "Citroën",
    "Almod?var": "Almodóvar",
    "Ana?s": "Anaïs",
    "Ladur?e": "Ladurée",
    "O?Connor": "O'Connor",
    "L'Or?al": "L'Oréal",
    "Hy?res": "Hyères",
    "Elys?e": "Elysée",
    "R?camier": "Récamier",
    "S?o": "São",
    "B?nstr?m": "Bönström",
    "B_nstr_m": "Bönström",
    "Fr_d_ric": "Frédéric",

    "Yigal AzrouФl": "Yigal Azrouël",
    "Kai KЩhne": "Kai Kühne",
    "St_rk": "Stärk",
    "Comme des GarЌons": "Comme des Garçons",
    "HermЏs": "Hermès",

    "Werkst?tte": "Werkstätte",
    "B?ndchen": "Bändchen",
    "D_coratifs": "Décoratifs",
    "R?yes": "Reyes",
    "Abaet_": "Abaeté",
    "Rhi_": "Rhié",

    # fashion / French terms
    "lam?": "lamé",
    "lam_": "lamé",
    "lam?s": "lamés",
    "Lam?s": "Lamés",

    "boucl?": "bouclé",
    "boucl_": "bouclé",
    "boucl?s": "bouclés",

    "pliss?": "plissé",
    "pliss_": "plissé",
    "pliss?s": "plissés",
    "pliss?e": "plissée",

    "piqu?": "piqué",
    "piqu_": "piqué",

    "soign?": "soigné",
    "soign_": "soigné",

    "ombr?": "ombré",
    "ombr_": "ombré",
    "ombr?-dyed": "ombré-dyed",
    "ombr?d": "ombréd",

    "d?j": "déjà",
    "tr?s": "très",
    "trs": "très",
    "apr?s-ski": "après-ski",
    "m?lange": "mélange",
    "m_lange": "mélange",
    "m?langes": "mélanges",
    "mise-en-sc?ne": "mise-en-scène",
    "na?ve": "naïve",
    "na?f": "naïf",
    "na?vet": "naïveté",
    "na«vet_": "naïveté",
    "Mus?e": "Musée",
    "Mus_e": "Musée",
    "H?tel": "Hôtel",
    "h?tel": "hôtel",
    "m?tier": "métier",
    "ma?tresse": "maîtresse",
    "si?cle": "siècle",
    "minaudi?re": "minaudière",
    "minaudi?res": "minaudières",
    "f?te": "fête",
    "marini?re": "marinière",
    "cr?me": "crème",
    "gr?ce": "grâce",
    "pi?ce": "pièce",
    "r?sistance": "résistance",
    "r?sum": "résumé",
    "caf?s": "cafés",
    "soir?e": "soirée",
    "fin-de-si?cle": "fin-de-siècle",
    "Proven?al": "Provençal",
    "d'?tre": "d'être",
    "d'tre": "d'être",
    "outr_": "outré",
    "risqu_": "risqué",
    "dor?e": "dorée",
    "pr?cis": "précis",
    "ing?nue": "ingénue",

    "appliqu?": "appliqué",
    "appliqu_": "appliqué",
    "appliqu?s": "appliqués",
    "appliqu_s": "appliqués",
    "appliqu?d": "appliquéd",
    "appliqu_d": "appliquéd",
    "appliqu_ing": "appliquéing",

    "d?colletage": "décolletage",
    "d?vor": "dévoré",
    "d_vor_": "dévoré",
    "devor_": "dévoré",
    "devor_s": "dévorés",
    "d?grad": "dégradé",
    "d_grad_": "dégradé",
    "clich?d": "clichéd",
    "clich_": "cliché",
    "clich?s": "clichés",
}

WORD_BRIDGE_MAP = {
    "?and": " and",
    "?a": " a",
    "?the": " the",
    "?but": " but",
    "?like": " like",
    "?in": " in",
    "?with": " with",
    "?was": " was",
    "?which": " which",
    "?even": " even",
    "?or": " or",
    "?to": " to",
    "?for": " for",
    "?as": " as",
    "?this": " this",
    "?that": " that",
    "?who": " who",
    "?it": " it",
    "?he": " he",
    "?she": " she",
    "?they": " they",
    "?we": " we",
    "?you": " you",
    "?are": " are",
    "?is": " is",
    "?were": " were",
    "?an": " an",
    "?one": " one",
    "?some": " some",
    "?most": " most",
    "?all": " all",
    "?those": " those",
    "?these": " these",
}

QUESTION_MARK_MAP = {
    r"(?<!\w)\?([a-zA-Z]+)(?!\w)": r"\1",
}



### Cleaning functions

In [10]:
# Cell 10: cleaning helper functions
def normalize_artifacts(text):
    if not isinstance(text, str):
        return text

    for bad, good in CONTROL_CHAR_MAP.items():
        text = text.replace(bad, good)

    # only convert underscores that are inside word characters
    text = re.sub(r"(?<=\w)_(?=\w)", "?", text)
    return text

def apply_literal_replacements(series, replacements):
    series = series.copy()

    for bad, good in replacements.items():
        series = series.str.replace(
            bad,
            good,
            regex=False
        )

    return series

def apply_regex_replacements(series, replacements):
    series = series.copy()

    for pattern, repl in replacements.items():
        series = series.str.replace(
            pattern,
            repl,
            regex=True,
            case=False
        )

    return series


In [11]:
# Cell 11: clean_text pipeline
def clean_text(series):
    """Cleans a pandas Series of text by applying a series of replacements and normalization"""

    series = series.copy()

    series = normalize_artifacts(series)
    series = apply_regex_replacements(series, CONTRACTION_MAP)
    series = apply_regex_replacements(series, NAME_MAP)
    series = apply_regex_replacements(series, DOMAIN_LANGUAGE_MAP)
    series = apply_literal_replacements(series, DIRECT_MAP)
    series = apply_literal_replacements(series, WORD_BRIDGE_MAP)
    return series

In [ ]:
# Cell 12: apply cleaning to columns
DIRTY_COLS = ["designer", "review"]

for col in DIRTY_COLS:
    df[f"{col}_clean"] = clean_text(df[col])

df.head()


## Normalize city column

In [ ]:
df["city"].unique()

In [ ]:
# Cell 13: city normalization map
LONG_CITY_MAP = {
    "London Moscow Istanbul Miami New York City Los Angeles Beirut London Copenhagen Paris Stockholm Dubai San Francisco Sydney Seoul S\xa0O Paulo Tokyo Beijing Mexico City Melbourne Versailles Kiev Munich Monaco Florence, Alabama Zurich Berlin Riyadh Milan Rio De Janeiro Las Vegas Palm Springs Nashville Shanghai Glasgow Turin Newcastle Upon Tyne Seattle Venice Odessa Negril Lagos Hebron Marrakech Cape Town Rome Gyeonggi-Do Abidjan St. Petersburg San Diego Kuwait City Manchester Lisbon Hong Kong Toronto Buenos Aires Chicago Guiyang Amsterdam Singapore Kassel Florence Mumbai Scottish Highlands Dallas Addis Ababa Golfe-Juan M_Lmo Marfa, Texas Abu Dhabi Anaheim, California Almaty Tel Aviv Amagansett Kyoto Phuket Edinburgh Tuscany Abraka Santa Barbara Patmos Majorca Abiquiu, New Mexico Ibiza Bangkok Nantes Barcelona Admont Bordeaux Manama Corsica Positano S\xa0O Luiz Do Paraitinga Portland Portofino Tarragona Washington D.C. Malibu Montauk Nikola-Lenivets Gotland Bergamo St. Tropez Lordville, New York Pushkinskiye Gory Capri New Orleans The Canary Islands Girona Carpathian Mountains Siem Reap": "London",

    "New York Moscow Istanbul Miami New York City Los Angeles Beirut London Copenhagen Paris Stockholm Dubai San Francisco Sydney Seoul S\xa0O Paulo Tokyo Beijing Mexico City Melbourne Versailles Kiev Munich Monaco Florence, Alabama Zurich Berlin Riyadh Milan Rio De Janeiro Las Vegas Palm Springs Nashville Shanghai Glasgow Turin Newcastle Upon Tyne Seattle Venice Odessa Negril Lagos Hebron Marrakech Cape Town Rome Gyeonggi-Do Abidjan St. Petersburg San Diego Kuwait City Manchester Lisbon Hong Kong Toronto Buenos Aires Chicago Guiyang Amsterdam Singapore Kassel Florence Mumbai Scottish Highlands Dallas Addis Ababa Golfe-Juan M_Lmo Marfa, Texas Abu Dhabi Anaheim, California Almaty Tel Aviv Amagansett Kyoto Phuket Edinburgh Tuscany Abraka Santa Barbara Patmos Majorca Abiquiu, New Mexico Ibiza Bangkok Nantes Barcelona Admont Bordeaux Manama Corsica Positano S\xa0O Luiz Do Paraitinga Portland Portofino Tarragona Washington D.C. Malibu Montauk Nikola-Lenivets Gotland Bergamo St. Tropez Lordville, New York Pushkinskiye Gory Capri New Orleans The Canary Islands Girona Carpathian Mountains Siem Reap": "New York",

    "Paris Moscow Istanbul Miami New York City Los Angeles Beirut London Copenhagen Paris Stockholm Dubai San Francisco Sydney Seoul S\xa0O Paulo Tokyo Beijing Mexico City Melbourne Versailles Kiev Munich Monaco Florence, Alabama Zurich Berlin Riyadh Milan Rio De Janeiro Las Vegas Palm Springs Nashville Shanghai Glasgow Turin Newcastle Upon Tyne Seattle Venice Odessa Negril Lagos Hebron Marrakech Cape Town Rome Gyeonggi-Do Abidjan St. Petersburg San Diego Kuwait City Manchester Lisbon Hong Kong Toronto Buenos Aires Chicago Guiyang Amsterdam Singapore Kassel Florence Mumbai Scottish Highlands Dallas Addis Ababa Golfe-Juan M_Lmo Marfa, Texas Abu Dhabi Anaheim, California Almaty Tel Aviv Amagansett Kyoto Phuket Edinburgh Tuscany Abraka Santa Barbara Patmos Majorca Abiquiu, New Mexico Ibiza Bangkok Nantes Barcelona Admont Bordeaux Manama Corsica Positano S\xa0O Luiz Do Paraitinga Portland Portofino Tarragona Washington D.C. Malibu Montauk Nikola-Lenivets Gotland Bergamo St. Tropez Lordville, New York Pushkinskiye Gory Capri New Orleans The Canary Islands Girona Carpathian Mountains Siem Reap": "Paris",

    "Milan Moscow Istanbul Miami New York City Los Angeles Beirut London Copenhagen Paris Stockholm Dubai San Francisco Sydney Seoul S\xa0O Paulo Tokyo Beijing Mexico City Melbourne Versailles Kiev Munich Monaco Florence, Alabama Zurich Berlin Riyadh Milan Rio De Janeiro Las Vegas Palm Springs Nashville Shanghai Glasgow Turin Newcastle Upon Tyne Seattle Venice Odessa Negril Lagos Hebron Marrakech Cape Town Rome Gyeonggi-Do Abidjan St. Petersburg San Diego Kuwait City Manchester Lisbon Hong Kong Toronto Buenos Aires Chicago Guiyang Amsterdam Singapore Kassel Florence Mumbai Scottish Highlands Dallas Addis Ababa Golfe-Juan M_Lmo Marfa, Texas Abu Dhabi Anaheim, California Almaty Tel Aviv Amagansett Kyoto Phuket Edinburgh Tuscany Abraka Santa Barbara Patmos Majorca Abiquiu, New Mexico Ibiza Bangkok Nantes Barcelona Admont Bordeaux Manama Corsica Positano S\xa0O Luiz Do Paraitinga Portland Portofino Tarragona Washington D.C. Malibu Montauk Nikola-Lenivets Gotland Bergamo St. Tropez Lordville, New York Pushkinskiye Gory Capri New Orleans The Canary Islands Girona Carpathian Mountains Siem Reap": "Milan",
}

In [ ]:
# Cell 14: normalize city values
df["city"] = df["city"].str.strip().str.title()
df["city"] = df["city"].replace(LONG_CITY_MAP)

df["city"].unique()

## Empty reviews

In [ ]:
# Cell 15: drop placeholder reviews
placeholder_pattern = re.compile(
    r"for more on this designer, click here"
    r"|please click here"
    r"|we['’]re posting runway pictures",
    flags=re.IGNORECASE
)

placeholder_mask = df["review_clean"].fillna("").apply(lambda x: bool(placeholder_pattern.search(x)))
print(f"Placeholder reviews found: {placeholder_mask.sum()}")

if placeholder_mask.sum() > 0:
    print(df[placeholder_mask][["designer_clean", "date", "review_clean"]].to_string())
    df = df[~placeholder_mask].reset_index(drop=True)
    print(f"\nDropped {placeholder_mask.sum()} placeholder rows. Remaining: {len(df)}")

# 4. Quality checks

## Remaining suspicious tokens

In [ ]:
# Cell 17: remaining suspicious tokens
remaining = Counter()
for col in ["designer_clean", "review_clean"]:
    for text in df[col].dropna():
        remaining.update(suspicious_tokens(text))

remaining_df = (
    pd.DataFrame(remaining.items(), columns=["token", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

remaining_df.to_csv(CLEANING_PATH / "last_remaining_suspect_tokens.csv", index=False)
remaining_df.head(200)

## Second-pass fixes

Derived from `last_remaining_suspect_tokens.csv` (tokens with count > 1 after the initial pipeline).
Applied as a separate step so the corrections are auditable and don't obscure the original cleaning logic.

In [ ]:
# Cell 21: second-pass maps
WORD_BRIDGE_MAP_EXT = {
    "\\?(?=\\S)": " "
}


SPECIFIC_FIXES_MAP = {
    "soup?on":       "soupçon",
    "soupon":        "soupçon",
    "Simo?ns":       "Simons",
    "B?nstr'm":      "Bönström",
    "Beyonc_":       "Beyoncé",
    "Dell?Acqua":    "Dell'Acqua",
    "Dell?Acqua's":  "Dell'Acqua's",
    "Dsquared?'s":   "DSquared2's",
    "Dsquared_":     "DSquared2",
    "Darr?'s":       "Darré's",
    "Sch?nberger":   "Schönberger",
    "Bu?uel":        "Buñuel",
    "Bu?uel's":      "Buñuel's",
    "Pen?lope":      "Penélope",
    "Ert_":          "Erté",
    "H_ch":          "Hüch",
    "Lang_s":        "Lang's",
    "he_s":          "he's",
    "d'I_na":        "d'Iéna",
    "d'I?na":        "d'Iéna",
    "Th_Štre":       "Théâtre",
    "M_bius":        "Möbius",
    "_l_gants":      "élégants",
    "Priv_":         "Privé",
    "Caf_":          "Café",
    "macram_":       "macramé",
    "macram?-ish":   "macramé-ish",
    "d_shabill_":    "déshabillé",
    "deshabill_":    "déshabillé",
    "d?tente":       "détente",
    "quincea?era":   "quinceañera",
    "d?fil":         "défilé",
    "ch?teau":       "château",
    "Ch?teau":       "Château",
    "papier-m?ch":   "papier-mâché",
    "coup_":         "coupé",
    "s?r":           "sûr",
    "protégé?s":     "protégé's",
    "l?oeil":        "l'œil",
    "late-?60s":     "late-'60s",
    "early-?60s":    "early-'60s",
    "late-?50s":     "late-'50s",
    "early-?80s":    "early-'80s",
    "mid-?60s":      "mid-'60s",
}

In [ ]:
# Cell 22: apply second-pass fixes and re-export
for col in ["designer_clean", "review_clean"]:
    df[col] = apply_regex_replacements(df[col], WORD_BRIDGE_MAP_EXT)
    df[col] = apply_literal_replacements(df[col], SPECIFIC_FIXES_MAP)

df_out = df[["year", "season", "designer_clean", "author", "city", "date", "review_clean"]].rename(columns={
    "designer_clean": "designer",
    "review_clean": "review",
})
df_out.to_csv("data/stylecom_cleaned.csv", index=False, encoding="utf-8")
print("Exported stylecom_cleaned.csv")

# 5. Write to clean csv


In [ ]:
# Cell 16: export cleaned CSV
df_out = df[["year", "season", "designer_clean", "author", "city", "date", "review_clean"]].rename(columns={
    "designer_clean": "designer",
    "review_clean": "review",
})

df_out.to_csv("data/stylecom_cleaned.csv", index=False, encoding="utf-8")

## Inspection

In [ ]:
# Cell 19: view reviews containing ?s
df[df["review_clean"].str.contains(r"\?s", na=False)]["review_clean"].head(10)

for x in df[df["review_clean"].str.contains(r"\?s", na=False)]["review_clean"].head(5):
    print(repr(x))